# Personal style LoRA: teach an open-weight LLM to write like me

Backbone for iteration 1. Runs on a rented 80 GB GPU (H100 or A100, Nebius or Google Cloud).

What this notebook does:
  1. Loads my exported messages (JSONL, my messages only, no context)
  2. Wraps each message as a chat turn: fixed instruction in, my message out
  3. Adds LoRA adapters to the last N decoder layers of one open-weight model
  4. Trains only on my message tokens (instruction tokens are masked out of the loss)
  5. Compares base vs adapted output on a few prompts
  6. Saves a versioned adapter that later runs can continue from

Model: `Qwen/Qwen3-8B` (Apache 2.0, no access request needed). Swap via
`TrainConfig.model_name`. `meta-llama/Llama-3.1-8B-Instruct` (gated, needs `HF_TOKEN`)
uses the same module names (`q_proj`, ..., `model.layers.<i>`).

Assumptions:
  - GPU with bf16 and 80 GB VRAM. Qwen3-8B weights take about 16 GB in bf16.
  - Messages are in English.
  - Messages file is JSONL, one object per line: `{"text": "<one of my messages>"}`
  - Same system prompt and instruction at training and inference time.

Limitations:
  - Without the message I replied to, the model learns how I write, not how I answer.
    Context pairs are a later iteration.
  - Random train/eval split: near-duplicate messages can land in both splits,
    so eval loss is optimistic.
  - Adapting only the last layers changes style (tone, length, word choice) more
    than knowledge. Do not expect it to learn facts about me reliably.

Privacy: the messages and the trained adapter (which can memorize messages) are
personal data. Keep both on the VM and in a private bucket, never in this repo or a
public Hugging Face repo. Cells print samples of the messages: clear all outputs
before committing this notebook.

Dependencies: all in `pyproject.toml`, nothing new.

## PART 0: Imports and GPU check

In [1]:
import hashlib
import json
import math
import os
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import (
    AutoConfig,
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    PreTrainedTokenizerBase,
    Trainer,
    TrainingArguments,
    set_seed,
)

/home/margot/projects/llm_xomnia_course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
assert torch.cuda.is_available(), (
    "No GPU found. Run this notebook on the rented GPU machine."
)
assert torch.cuda.is_bf16_supported(), (
    "This GPU has no bf16 support (e.g. T4, V100). Use an H100 or A100."
)

props = torch.cuda.get_device_properties(0)
print(f"GPU:   {props.name}")
print(f"VRAM:  {props.total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__}, CUDA: {torch.version.cuda}")

AssertionError: No GPU found. Run this notebook on the rented GPU machine.

## PART 1: Configuration and resource paths

Every path is a dummy. Each one can be overridden with an environment variable, so the
same notebook runs on Nebius and Google Cloud without edits. Set them before starting
Jupyter, or at the top of this notebook with `%env NAME=value`.

| Resource | Env var | Dummy default | Notes |
|---|---|---|---|
| Work root | `STYLE_LORA_ROOT` | `/mnt/data/style-lora` | Persistent disk on the GPU VM. Nebius: mounted network disk or shared filesystem. Google Cloud: attached persistent disk (Vertex AI Workbench: under `/home/jupyter`). |
| Messages | `STYLE_LORA_MESSAGES` | `<root>/data/my_messages.jsonl` | My export, one `{"text": ...}` per line |
| Model cache | `STYLE_LORA_HF_CACHE` | `<root>/hf_cache` | About 16 GB for Qwen3-8B. On persistent disk so restarts do not re-download |
| Runs | `STYLE_LORA_RUNS` | `<root>/runs` | One folder per run: `checkpoints/`, `adapter/`, `run_config.json` |
| Previous adapter | `STYLE_LORA_INIT_ADAPTER` | unset | Re-train from an earlier run: `<root>/runs/<run_name>/adapter` |
| Run name | `STYLE_LORA_RUN_NAME` | generated | Set to an existing run name to resume that run |
| Resume | `STYLE_LORA_RESUME` | `0` | `1` resumes the latest checkpoint of `STYLE_LORA_RUN_NAME` (VM was stopped or preempted) |
| Bucket | `STYLE_LORA_BUCKET` | `s3://<nebius-bucket>/style-lora` | Durable copy of data and runs. Google Cloud: `gs://<gcs-bucket>/style-lora`. See PART 10 |
| HF token | `HF_TOKEN` | unset | Only for gated models (Llama, Gemma). Qwen3 is ungated. Never write it in the notebook |

Example on the VM:

```bash
export STYLE_LORA_ROOT=/mnt/data/style-lora
export STYLE_LORA_BUCKET=gs://<gcs-bucket>/style-lora   # or s3://<nebius-bucket>/style-lora
```

In [ ]:
def env_path(name: str, default: Path) -> Path:
    """Read a path from an environment variable, else use the dummy default."""
    return Path(os.environ.get(name, str(default)))


def env_optional_path(name: str) -> Path | None:
    """Read an optional path from an environment variable; None when unset."""
    value = os.environ.get(name, "").strip()
    return Path(value) if value else None


ROOT = env_path("STYLE_LORA_ROOT", Path("/mnt/data/style-lora"))


@dataclass(frozen=True)
class Paths:
    """Resource locations on the GPU machine; dummies unless overridden."""

    messages: Path = env_path(
        "STYLE_LORA_MESSAGES", ROOT / "data" / "my_messages.jsonl"
    )
    hf_cache: Path = env_path("STYLE_LORA_HF_CACHE", ROOT / "hf_cache")
    runs: Path = env_path("STYLE_LORA_RUNS", ROOT / "runs")
    init_adapter: Path | None = env_optional_path("STYLE_LORA_INIT_ADAPTER")
    bucket: str = os.environ.get(
        "STYLE_LORA_BUCKET", "s3://<nebius-bucket>/style-lora"
    )


@dataclass(frozen=True)
class TrainConfig:
    """Model, data, LoRA and optimization settings."""

    # Model and chat format
    model_name: str = "Qwen/Qwen3-8B"
    system_prompt: str = (
        "You write messages in my personal style: same tone, length, "
        "vocabulary and punctuation."
    )
    user_prompt: str = "Write a message."  # fixed instruction for every example
    max_length: int = 512
    eval_fraction: float = 0.1
    seed: int = 42

    # LoRA: adapters only in the last n_last_layers decoder blocks
    n_last_layers: int = 8
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    target_modules: tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    # Optimization, sized for an 80 GB GPU
    n_epochs: int = 3
    learning_rate: float = 2e-4
    per_device_batch_size: int = 8
    grad_accum_steps: int = 2
    warmup_fraction: float = 0.05
    weight_decay: float = 0.0
    gradient_checkpointing: bool = False  # set True if you hit OOM
    eval_steps: int = 50
    save_steps: int = 50
    logging_steps: int = 10


paths = Paths()
cfg = TrainConfig()
set_seed(cfg.seed)

# Reuse an existing run name to resume it; otherwise a new versioned name
RUN_NAME = os.environ.get("STYLE_LORA_RUN_NAME") or (
    f"{cfg.model_name.split('/')[-1].lower()}"
    f"-last{cfg.n_last_layers}-r{cfg.lora_r}"
    f"-{datetime.now(timezone.utc):%Y%m%d-%H%M%S}"
)
RESUME = os.environ.get("STYLE_LORA_RESUME", "0") == "1"

run_dir = paths.runs / RUN_NAME
checkpoint_dir = run_dir / "checkpoints"
adapter_dir = run_dir / "adapter"

# Prompts for the side-by-side check in PART 8. The first is the training
# instruction; the others test whether the style carries over to new requests.
EVAL_PROMPTS = [
    cfg.user_prompt,
    "Write a message to your team: the sprint demo moves to Thursday.",
    "Write a reply to a colleague who asks if you can review their PR today.",
]

for name, value in asdict(paths).items():
    print(f"{name:<14} {value}")
print(f"{'run_dir':<14} {run_dir}")
print(f"{'resume':<14} {RESUME}")
print(f"{'HF_TOKEN set':<14} {bool(os.environ.get('HF_TOKEN'))}")

## PART 2: Load and validate messages

Validate before spending GPU time: schema, empty rows, duplicates, size.

In [ ]:
TEXT_COLUMN = "text"
MIN_RECOMMENDED_ROWS = 500


def load_messages(path: Path) -> pd.DataFrame:
    """Load the JSONL messages export and check the schema."""
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Set STYLE_LORA_MESSAGES or copy the file there."
        )
    df = pd.read_json(path, lines=True, dtype=False)
    if TEXT_COLUMN not in df.columns:
        raise ValueError(f"{path} has no '{TEXT_COLUMN}' field: {list(df.columns)}")
    return df[[TEXT_COLUMN]]


def clean_messages(df: pd.DataFrame) -> pd.DataFrame:
    """Strip whitespace, drop empty messages and exact duplicates."""
    text = df[TEXT_COLUMN].fillna("").astype(str).str.strip()
    text = text[text.str.len() > 0].drop_duplicates()
    return text.reset_index(drop=True).to_frame()


raw_df = load_messages(paths.messages)
messages_df = clean_messages(raw_df)
n_chars = messages_df[TEXT_COLUMN].str.len()

print(f"Rows loaded:           {len(raw_df):,}")
print(f"Rows after cleaning:   {len(messages_df):,}")
print(f"Shorter than 10 chars: {(n_chars < 10).mean():.1%}")
print(f"Chars p50 / p90 / max: {n_chars.median():.0f} / "
      f"{n_chars.quantile(0.9):.0f} / {n_chars.max()}")
if len(messages_df) < MIN_RECOMMENDED_ROWS:
    print(f"WARNING: fewer than {MIN_RECOMMENDED_ROWS} messages, expect overfitting.")

messages_df.sample(min(5, len(messages_df)), random_state=cfg.seed)

In [ ]:
# Split before tokenization; seeded so re-runs give the same eval set
split = Dataset.from_pandas(messages_df, preserve_index=False).train_test_split(
    test_size=cfg.eval_fraction, seed=cfg.seed
)
print(split)

## PART 3: Chat formatting and loss masking

Each message becomes one conversation:

```
system:    <style instruction>        masked (label = -100)
user:      Write a message.           masked
assistant: <my message><end of turn>  trained
```

The model's own chat template is used, so the adapter works with normal chat
inference afterwards. Qwen3 thinking mode is disabled: my messages have no
reasoning trace.

In [ ]:
IGNORE_INDEX = -100  # label value skipped by the cross-entropy loss

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, cache_dir=paths.hf_cache)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def build_messages(
    cfg: TrainConfig, instruction: str, message: str | None = None
) -> list[dict[str, str]]:
    """Build system/user(/assistant) turns; the assistant turn is my message."""
    turns = [
        {"role": "system", "content": cfg.system_prompt},
        {"role": "user", "content": instruction},
    ]
    if message is not None:
        turns.append({"role": "assistant", "content": message})
    return turns


def render_chat(
    tokenizer: PreTrainedTokenizerBase,
    turns: list[dict[str, str]],
    add_generation_prompt: bool,
) -> str:
    """Render turns with the model's chat template, thinking disabled."""
    # enable_thinking is a Qwen3 template variable; other templates ignore it
    return tokenizer.apply_chat_template(
        turns,
        tokenize=False,
        add_generation_prompt=add_generation_prompt,
        enable_thinking=False,
    )


def tokenize_example(
    row: dict[str, str], tokenizer: PreTrainedTokenizerBase, cfg: TrainConfig
) -> dict[str, list[int]]:
    """Tokenize one message and mask every token except the message itself."""
    prompt_text = render_chat(
        tokenizer, build_messages(cfg, cfg.user_prompt), add_generation_prompt=True
    )
    full_text = render_chat(
        tokenizer,
        build_messages(cfg, cfg.user_prompt, row[TEXT_COLUMN]),
        add_generation_prompt=False,
    )
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]
    if full_ids[: len(prompt_ids)] != prompt_ids:
        raise ValueError(
            "Chat template mismatch: the prompt is not a prefix of the full "
            "conversation. Check the template after swapping models."
        )
    labels = [IGNORE_INDEX] * len(prompt_ids) + full_ids[len(prompt_ids):]
    full_ids, labels = full_ids[: cfg.max_length], labels[: cfg.max_length]
    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels,
    }

In [ ]:
tokenized = split.map(
    tokenize_example,
    fn_kwargs={"tokenizer": tokenizer, "cfg": cfg},
    remove_columns=[TEXT_COLUMN],
    desc="Tokenizing",
)
n_before = {name: len(ds) for name, ds in tokenized.items()}
# Drop examples whose message was cut off entirely by max_length
tokenized = tokenized.filter(
    lambda row: any(t != IGNORE_INDEX for t in row["labels"])
)
train_ds, eval_ds = tokenized["train"], tokenized["test"]

lengths = np.array([len(ids) for ids in train_ds["input_ids"]])
print(f"Train examples: {len(train_ds):,} "
      f"(dropped {n_before['train'] - len(train_ds)})")
print(f"Eval examples:  {len(eval_ds):,} "
      f"(dropped {n_before['test'] - len(eval_ds)})")
print(f"Tokens per example: p50={np.percentile(lengths, 50):.0f}, "
      f"p90={np.percentile(lengths, 90):.0f}, max={lengths.max()}")
print(f"Truncated at max_length={cfg.max_length}: "
      f"{(lengths == cfg.max_length).mean():.1%}")

In [ ]:
# The loss should cover only my message and the end-of-turn token
example = train_ds[0]
print("FULL INPUT:")
print(tokenizer.decode(example["input_ids"]))
print()
print("TRAINED ON:")
print(tokenizer.decode([t for t in example["labels"] if t != IGNORE_INDEX]))

## PART 4: Load the base model

bf16 weights, whole model on GPU 0. The number of decoder layers is read from the
model config, so the last-N selection stays correct after swapping models.

In [ ]:
model_config = AutoConfig.from_pretrained(cfg.model_name, cache_dir=paths.hf_cache)
n_layers = model_config.num_hidden_layers
last_layer_ids = list(range(n_layers - cfg.n_last_layers, n_layers))

base_model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    dtype=torch.bfloat16,
    device_map={"": 0},
    cache_dir=paths.hf_cache,
)
base_model.config.use_cache = False  # no KV cache needed during training

n_params = sum(p.numel() for p in base_model.parameters())
print(f"Parameters:        {n_params / 1e9:.2f} B")
print(f"Decoder layers:    {n_layers}")
print(f"Layers to adapt:   {last_layer_ids}")
print(f"GPU memory (load): {torch.cuda.memory_allocated() / 1e9:.1f} GB")

## PART 5: LoRA on the last N layers

`layers_to_transform` restricts LoRA to the listed decoder blocks. `target_modules`
picks the linear layers inside each block (attention and MLP).

Why the last layers:
  - Working assumption, to be tested by ablation: late layers shape token choice
    and tone, early layers hold general language features we want to keep.
  - Memory and speed: no layer before the first adapter has trainable parameters,
    so autograd builds no graph there. Backward runs through N layers instead of
    all of them, and only those N layers keep activations.

Expected for Qwen3-8B, last 8 layers, r=16, all 7 projections: about 9.7M trainable
parameters (about 0.1%). `print_trainable_parameters` below confirms it.

Re-train mode (`STYLE_LORA_INIT_ADAPTER` set): the previous adapter is loaded as
trainable and its saved LoRA settings apply. `n_last_layers` and `lora_*` are ignored.

In [ ]:
def build_lora_config(cfg: TrainConfig, layer_ids: list[int]) -> LoraConfig:
    """LoRA config restricted to the given decoder layer indices."""
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=list(cfg.target_modules),
        layers_to_transform=layer_ids,
        layers_pattern="layers",  # module path is model.layers.<i>.<...>
        bias="none",
    )


def adapted_layer_ids(model: torch.nn.Module) -> list[int]:
    """Return the decoder layer indices that hold trainable parameters."""
    ids = {
        int(name.split(".layers.")[1].split(".")[0])
        for name, param in model.named_parameters()
        if param.requires_grad and ".layers." in name
    }
    return sorted(ids)


if paths.init_adapter is not None:
    # Re-train: continue from a previous run's adapter
    model = PeftModel.from_pretrained(
        base_model, str(paths.init_adapter), is_trainable=True
    )
else:
    model = get_peft_model(base_model, build_lora_config(cfg, last_layer_ids))

model.print_trainable_parameters()
adapted_layers = adapted_layer_ids(model)
print(f"Layers with adapters: {adapted_layers}")
if paths.init_adapter is None:
    assert adapted_layers == last_layer_ids, "LoRA landed on unexpected layers"

## PART 6: Train

Hugging Face `Trainer` handles gradient accumulation, bf16 autocast, the cosine
schedule with warmup (stepped per optimizer step), gradient clipping, checkpoints
and best-checkpoint selection on eval loss.

Metric: eval loss (and perplexity) on held-out messages, my tokens only.
Baseline: the same loss before training. With a fresh adapter (B = 0) this equals
the base model; in re-train mode it is the previous adapter.

In [ ]:
effective_batch = cfg.per_device_batch_size * cfg.grad_accum_steps
total_steps = math.ceil(len(train_ds) / effective_batch) * cfg.n_epochs
warmup_steps = max(1, round(cfg.warmup_fraction * total_steps))

training_args = TrainingArguments(
    output_dir=str(checkpoint_dir),
    num_train_epochs=cfg.n_epochs,
    per_device_train_batch_size=cfg.per_device_batch_size,
    per_device_eval_batch_size=cfg.per_device_batch_size,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    learning_rate=cfg.learning_rate,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    weight_decay=cfg.weight_decay,
    max_grad_norm=1.0,
    bf16=True,
    gradient_checkpointing=cfg.gradient_checkpointing,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    eval_strategy="steps",
    eval_steps=cfg.eval_steps,
    save_strategy="steps",
    save_steps=cfg.save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=cfg.logging_steps,
    report_to="none",  # "mlflow" or "wandb" once tracking is set up
    seed=cfg.seed,
    data_seed=cfg.seed,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        padding=True,
        label_pad_token_id=IGNORE_INDEX,
        pad_to_multiple_of=8,
    ),
    processing_class=tokenizer,
)

print(f"Effective batch size: {effective_batch}")
print(f"Optimizer steps:      {total_steps} (warmup {warmup_steps})")

In [ ]:
baseline_metrics = trainer.evaluate()
print(f"Baseline eval loss: {baseline_metrics['eval_loss']:.4f} "
      f"(perplexity {math.exp(baseline_metrics['eval_loss']):.1f})")

In [ ]:
torch.cuda.reset_peak_memory_stats()
train_result = trainer.train(resume_from_checkpoint=RESUME or None)
final_metrics = trainer.evaluate()  # best checkpoint is loaded at the end

print(f"Train runtime:      {train_result.metrics['train_runtime'] / 60:.1f} min")
print(f"Peak GPU memory:    {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
print(f"Baseline eval loss: {baseline_metrics['eval_loss']:.4f}")
print(f"Final eval loss:    {final_metrics['eval_loss']:.4f} "
      f"(perplexity {math.exp(final_metrics['eval_loss']):.1f})")

## PART 7: Training curves

Eval loss rising while train loss keeps falling means the adapter is memorizing
messages: fewer epochs, lower `lora_r`, or more data.

In [ ]:
history = trainer.state.log_history
train_points = [(h["step"], h["loss"]) for h in history if "loss" in h]
eval_points = [(h["step"], h["eval_loss"]) for h in history if "eval_loss" in h]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(*zip(*train_points), label="Train loss")
ax.plot(*zip(*eval_points), marker="o", label="Eval loss")
ax.axhline(baseline_metrics["eval_loss"], color="gray", linestyle="--",
           label="Baseline eval loss")
ax.set_xlabel("Optimizer step")
ax.set_ylabel("Loss (my tokens only)")
ax.set_title("Style LoRA training")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## PART 8: Base vs adapted output

Same model object: `disable_adapter()` switches LoRA off for the base output.
Same seed for both, so differences come from the adapter, not sampling noise.

In [ ]:
def generate_reply(
    model: PeftModel,
    tokenizer: PreTrainedTokenizerBase,
    cfg: TrainConfig,
    instruction: str,
    max_new_tokens: int = 128,
) -> str:
    """Generate one sampled message for an instruction in the training format."""
    text = render_chat(
        tokenizer, build_messages(cfg, instruction), add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)
    inputs = inputs.to(model.device)
    set_seed(cfg.seed)
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


model.eval()
for instruction in EVAL_PROMPTS:
    with model.disable_adapter():
        base_reply = generate_reply(model, tokenizer, cfg, instruction)
    adapted_reply = generate_reply(model, tokenizer, cfg, instruction)
    print(f"PROMPT:  {instruction}")
    print(f"BASE:    {base_reply}")
    print(f"ADAPTED: {adapted_reply}")
    print()

## PART 9: Save the adapter and run record

Only the adapter is saved (tens of MB), not the 16 GB base model. `run_config.json`
records config, data hash and metrics, so every run can be traced back to the exact
messages file it was trained on.

In [ ]:
run_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(adapter_dir))  # adapter weights and tokenizer only

run_record = {
    "run_name": RUN_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "messages_sha256": hashlib.sha256(paths.messages.read_bytes()).hexdigest(),
    "paths": asdict(paths),
    "config": asdict(cfg),
    "adapted_layers": adapted_layers,
    "n_train": len(train_ds),
    "n_eval": len(eval_ds),
    "baseline_eval_loss": baseline_metrics["eval_loss"],
    "final_eval_loss": final_metrics["eval_loss"],
    "peak_gpu_memory_gb": torch.cuda.max_memory_allocated() / 1e9,
}
(run_dir / "run_config.json").write_text(
    json.dumps(run_record, indent=2, default=str)
)

print(f"Saved run to {run_dir}")
for f in sorted(adapter_dir.iterdir()):
    print(f"  {f.name:<35} {f.stat().st_size / 1e6:>8.2f} MB")

## PART 10: Copy data and runs to durable storage

Run these in a terminal, not from the notebook. Replace the placeholders.

Before training: upload the messages once from your laptop, then pull them onto the VM.

```bash
# Nebius Object Storage (S3-compatible, AWS CLI)
aws s3 cp my_messages.jsonl s3://<nebius-bucket>/style-lora/data/ --endpoint-url https://storage.<region>.nebius.cloud
aws s3 cp s3://<nebius-bucket>/style-lora/data/my_messages.jsonl /mnt/data/style-lora/data/ --endpoint-url https://storage.<region>.nebius.cloud

# Google Cloud Storage
gcloud storage cp my_messages.jsonl gs://<gcs-bucket>/style-lora/data/
gcloud storage cp gs://<gcs-bucket>/style-lora/data/my_messages.jsonl /mnt/data/style-lora/data/
```

After training: push the run folder so it survives deleting the VM.

```bash
aws s3 sync /mnt/data/style-lora/runs/<run_name> s3://<nebius-bucket>/style-lora/runs/<run_name> --endpoint-url https://storage.<region>.nebius.cloud
gcloud storage rsync --recursive /mnt/data/style-lora/runs/<run_name> gs://<gcs-bucket>/style-lora/runs/<run_name>
```

## PART 11: Re-training and next iterations

| Goal | Set | Then |
|---|---|---|
| Continue an interrupted run | `STYLE_LORA_RUN_NAME=<run_name>`, `STYLE_LORA_RESUME=1` | Restart kernel, run all |
| Train further on new or more messages | `STYLE_LORA_INIT_ADAPTER=<root>/runs/<run_name>/adapter`, point `STYLE_LORA_MESSAGES` at the new file | Restart kernel, run all. A new run folder is created |
| Fresh run with other settings | Edit `TrainConfig` | Restart kernel, run all |

Leakage across re-trains: with a changed messages file, the random split changes, so
messages the previous adapter trained on can land in the new eval set. Fix with a
frozen eval file in a later iteration.

Next iterations (not in this backbone):
  - Frozen eval file across runs; bootstrap a confidence interval on eval loss
  - Ablate `n_last_layers` (4, 8, 16, all) and `lora_r` against eval loss
  - Context pairs (message I replied to + my reply) so the model learns how I answer
  - Blind test: mix real and generated messages, check whether they can be told apart
  - Experiment tracking: `report_to="mlflow"` or `"wandb"`
  - Serving: `merge_and_unload()` or load the adapter in vLLM